In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Projects/Medical_LMM/
!pwd

/content/drive/MyDrive/Projects/Medical_LMM
/content/drive/MyDrive/Projects/Medical_LMM


In [ ]:
!pip install -U bitsandbytes
!python -m pip install -U matplotlib
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 61.1 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 36.8 MB/s eta 0:00:00


In [ ]:
import torch
import os
import json
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

original_svd = torch.linalg.svd

def hybrid_svd(A, full_matrices=True, driver=None, *args, **kwargs):
    if torch.cuda.is_available():
        A_compute = A.to(device="cuda", dtype=torch.float32)
    else:
        A_compute = A.float()

    U, S, Vh = original_svd(A_compute, full_matrices=full_matrices, driver=driver, *args, **kwargs)

    # Move result back to CPU immediately to free VRAM
    return (
        U.to(device="cpu", dtype=torch.bfloat16),
        S.to(device="cpu", dtype=torch.bfloat16),
        Vh.to(device="cpu", dtype=torch.bfloat16)
    )

torch.linalg.svd = hybrid_svd

model_name = "Qwen/Qwen3-8B"
save_dir_base = "./princeps/model/Qwen-Base"
save_dir_adapter = "./princeps/model/Qwen-QPiSSA-Adapter"

print(f"Loading {model_name} on CPU... (System RAM)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map={"": "cpu"},
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Running PiSSA SVD...")
start_time = time.time()

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=32,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    init_lora_weights="pissa"
)

model = get_peft_model(model, peft_config)

print(f"SVD Complete in {time.time() - start_time:.2f} seconds.")

print("Saving PiSSA Adapters...")
model.save_pretrained(save_dir_adapter)

print("Saving Residual Base Model...")
unwrapped_model = model.unload()

unwrapped_model.save_pretrained(save_dir_base, safe_serialization=True)
tokenizer.save_pretrained(save_dir_base)

config_path = os.path.join(save_dir_adapter, "adapter_config.json")
with open(config_path, "r") as f:
    config = json.load(f)

config["init_lora_weights"] = False
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Decomposition Complete!")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


model_id = "Qwen/Qwen3-8B"

case_text = """
A 45-year-old man from a rural village in Bolivia presents with a 10-year history of progressive difficulty swallowing (dysphagia) and frequent regurgitation.
He also reports chronic constipation. A physical exam is unremarkable, but a barium swallow study reveals a significantly enlarged esophagus (megaesophagus)
with poor peristalsis. He has no history of heart disease.
"""

quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

print(f"Loading {model_id} in 8-bit precision...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True
    )
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

messages = [
    {"role": "system", "content": "You are a helpful medical assistant. Analyze the medical presentatio and provide a diagnosis along with your rationale."},
    {"role": "user", "content": f"Patient Case:\n{case_text}"}
]

text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text_input], return_tensors="pt").to("cuda")

print("\n--- Generating Response ---\n")

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2048,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1
    )


generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("-" * 30)
print(response)
print("-" * 30)

Loading Qwen/Qwen3-8B in 8-bit precision...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


--- Generating Response ---

------------------------------
<think>
Okay, let's tackle this patient case. So, the patient is a 45-year-old man from a rural area in Bolivia. He's had progressive dysphagia and frequent regurgitation for 10 years, plus chronic constipation. The physical exam is normal, but the barium swallow shows megaesophagus with poor peristalsis. No heart disease history.

First, I need to think about causes of megaesophagus. Megaesophagus can be due to motility disorders, structural issues, or systemic diseases. Since there's poor peristalsis, it's likely a motility problem. Let me recall the common conditions.

Chagas disease comes to mind because it's endemic in parts of South America, including Bolivia. Chagas is caused by Trypanosoma cruzi. The cardiac form is more well-known, but the gastrointestinal form can cause megaesophagus. It usually starts with acute symptoms like fever and swelling, then a latent period, followed by chronic complications. In the GI tra

In [ ]:
import torch
import json
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm


model_id = "./princeps/model/Qwen-Base"
adapter_id = "./princeps/inference-ready/Prime/Qwen-QPiSSA-Adapter-FT"
input_file = "val-data.json"
output_file = "single-case-result.json"


quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

print("Loading base in 8-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)


print("Plugging in the fine-tuned adapter...")
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()


with open(input_file, "r") as f:
    full_data = json.load(f)


if len(full_data) > 0:
    val_data = full_data[:1]  # This takes only the first element
    print(f"Dataset sliced. Processing only the first case (ID: {val_data[0].get('id', 'Unknown')})")
else:
    print("Error: Validation file is empty.")
    exit()

results = []

for case in tqdm(val_data):
    case_text = case.get("prompt", "")
    prompt = (
        f"Analyze the clinical presentation and provide a diagnosis.\n\n"
        f"Patient Case:\n{case_text}\n\n"
        f"Rationale:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )


    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    print("\n--- Model Output ---")
    print(generated_text)
    print("--------------------\n")


    results.append({
        "id": case.get("id"),
        "input_case": case_text,
        "expected_diagnosis": case.get("diagnosis") or case.get("expected_diagnosis"),
        "model_output": generated_text.strip()
    })


with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print(f"Inference complete! Results saved to {output_file}")

Loading residual base in 8-bit...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Plugging in the fine-tuned adapter...
Dataset sliced. Processing only the first case (ID: case-001)


100%|██████████| 1/1 [00:33<00:00, 33.57s/it]


--- Model Output ---
Rationale: The patient's long-standing dysphagia, regurgitation, constipation, and megaesophagus strongly suggest an underlying neuromuscular disorder affecting esophageal motility. Given his origin from Bolivia, where Chagas disease is endemic, this raises suspicion for Chagas cardiomyopathy as a cause of autonomic dysfunction leading to megaesophagus. While other conditions like scleroderma or achalasia can cause similar symptoms, the geographic location and prolonged course make Chagas disease the most likely diagnosis due to its association with both cardiac involvement and gastrointestinal dysmotility presenting as megaesophagus.

Diagnosis:
Chagas Disease
--------------------

Inference complete! Results saved to single-case-result.json


In [ ]:
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import gc

# --- Configuration ---
residual_base_path = "./princeps/model/Qwen-Base"
adapter_path = "./princeps/model/Qwen-QPiSSA-Adapter"
output_dir = "./princeps/inference-ready/Qwen-QPiSSA-Adapter-FT"

data_file = "distilled-data.json"
validation_data = "val-data.json"

# --- 1. Load Data & Create Validation Split ---
print("--- Loading Data ---")
dataset = load_dataset("json", data_files=data_file, split="train")
eval_dataset = load_dataset("json", data_files=validation_data, split="train")

train_dataset = dataset.rename_column("prompt", "case_text")
eval_dataset = eval_dataset.rename_column("prompt", "case_text")

print(f"Training on {len(train_dataset)} samples")
print(f"Validating on {len(eval_dataset)} samples")

# --- 2. Load Residual Base (4-bit) ---
print("--- Loading Residual Base (4-bit) ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    residual_base_path,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)

model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(residual_base_path)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 3. Load PiSSA Adapters ---
print("--- Loading PiSSA Adapters ---")
model = PeftModel.from_pretrained(
    model,
    adapter_path,
    is_trainable=True
)

model.print_trainable_parameters()

# --- 4. Training Arguments ---
print("--- Starting QPiSSA Training ---")

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    num_train_epochs=8,
    logging_steps=5,
    optim="paged_adamw_32bit",
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,

    eval_strategy="steps",
    save_strategy="steps",
    eval_steps=60,
    save_steps=60,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # Watch the validation loss
    greater_is_better=False,

    weight_decay=0.2,
    warmup_ratio=0.1,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    max_length=1024,
    dataset_text_field="text",
    packing=False,
    report_to="none"
)

# --- 5. Pre-Processing & Masking Logic (Replaces Formatter) ---
def process_and_mask(sample):
    prompt_text = (
        f"Analyze the clinical presentation and provide a diagnosis.\n\n"
        f"Patient Case:\n{sample['case_text']}\n\n"
    )

    completion_text = (
        f"Diagnosis:\n{sample['diagnosis']}"
        f"{tokenizer.eos_token}"
    )

    # add_special_tokens=True adds BOS to the prompt
    prompt_ids = tokenizer(prompt_text, add_special_tokens=True).input_ids
    completion_ids = tokenizer(completion_text, add_special_tokens=False).input_ids

    # Concatenate
    input_ids = prompt_ids + completion_ids

    # Create Labels: -100 masks the loss for prompt tokens
    prompt_mask = [-100] * len(prompt_ids)
    labels = prompt_mask + completion_ids

    # Optional: Truncate to max_length
    max_len = training_args.max_length
    if len(input_ids) > max_len:
        input_ids = input_ids[:max_len]
        labels = labels[:max_len]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels
    }

print("--- Pre-processing and Masking Data ---")
train_dataset = train_dataset.map(process_and_mask, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(process_and_mask, remove_columns=eval_dataset.column_names)

# --- 6. Initialize SFTTrainer ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),

    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

print("--- Saving Final Model ---")
trainer.save_model(output_dir)

history = trainer.state.log_history

print("Done Training!")

# --- 8. Plotting ---
train_steps = [x['step'] for x in history if 'loss' in x]
train_loss = [x['loss'] for x in history if 'loss' in x]

eval_steps = [x['step'] for x in history if 'eval_loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_loss, label='Training Loss', color='blue')
if eval_loss:
    plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red', linestyle='--')

plt.title('Training vs Validation Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Save plot
plt.savefig(f"{output_dir}/loss_curve.png")
print(f"Graph saved to {output_dir}/loss_curve.png")

--- Loading Data ---


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Training on 107 samples
Validating on 20 samples
--- Loading Residual Base (4-bit) ---


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


Loading unquantized residual base from: ./princeps/model/Qwen-QPiSSA-Residual-Base


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading fine-tuned adapter from: ./princeps/inference-ready/Qwen-QPiSSA-Adapter-FT


ValueError: We need an `offload_dir` to dispatch this model according to this `device_map`, the following submodules need to be offloaded: base_model.model.model.layers.19, base_model.model.model.layers.20, base_model.model.model.layers.21, base_model.model.model.layers.22, base_model.model.model.layers.23, base_model.model.model.layers.24, base_model.model.model.layers.25, base_model.model.model.layers.26, base_model.model.model.layers.27, base_model.model.model.layers.28, base_model.model.model.layers.29, base_model.model.model.layers.30, base_model.model.model.layers.31, base_model.model.model.layers.32, base_model.model.model.layers.33, base_model.model.model.layers.34, base_model.model.model.layers.35, base_model.model.model.norm, base_model.model.model.rotary_emb, base_model.model.lm_head.

In [ ]:
import torch
import json
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

model_id = "./princeps/model/Qwen-Base"
adapter_id = "./princeps/inference-ready/Prime/Qwen-QPiSSA-Adapter-FT"
input_file = "val-data.json"
output_file = "val-res.json"

# Setup 8-bit configuration
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)

print("Loading residual base in 8-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

# Attach the Adapter
print("Plugging in the fine-tuned adapter...")
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()

with open(input_file, "r") as f:
    val_data = json.load(f)

results = []

print(f"Starting inference on {len(val_data)} cases...")

# 5. Batch Processing Loop
for case in tqdm(val_data):
    case_text = case.get("prompt", "")
    prompt = (
        f"Analyze the clinical presentation and provide a diagnosis.\n\n"
        f"Patient Case:\n{case_text}\n\n"
        f"Rationale:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,  # Low temperature for deterministic medical reasoning
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    results.append({
        "id": case.get("id"),
        "input_case": case_text,
        "expected_diagnosis": case.get("diagnosis") or case.get("expected_diagnosis"),
        "model_output": generated_text.strip()
    })

# 6. Save Results
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print(f"Inference complete! Results saved to {output_file}")

Loading residual base in 8-bit...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Plugging in the fine-tuned adapter...
Starting inference on 20 cases...


100%|██████████| 20/20 [08:04<00:00, 24.22s/it]


Inference complete! Results saved to val-res_experior.json


In [ ]:
from google.colab import runtime
runtime.unassign()